# 🏥 LabLens AI — Cloud GPU Runner (Zero-Token Gemma 4 12B Pipeline)
### 100% Free, Zero Hugging Face Login & Zero Token Required

**Team:** @Abhishekdabi33, @gunwantjawle-ux, @vedxnt-b, @piyushmatodepatil  
**Platform:** Kaggle Dual Tesla T4 GPUs (32GB Total VRAM)  
**Engine:** Native Ollama + Google Gemma 4 12B (Clinical Review & EvidenceTrace)  

---

## Cell 1: Check GPU Acceleration
Make sure **GPU T4 x2** is selected in Kaggle (Settings -> Accelerator -> GPU T4 x2).

In [ ]:
!nvidia-smi

## Cell 2: Install zstd & Native Ollama Engine

In [ ]:
# 1. Install zstd (required by Kaggle Ubuntu)
!apt-get update -qq && apt-get install -y -qq zstd

# 2. Install official Ollama binary
!curl -fsSL https://ollama.com/install.sh | sh

# 3. Install testing and tunnel libraries
!pip install -q pyngrok httpx pydantic

# 4. Confirm installation version
!ollama --version
print("✅ Native Ollama engine and dependencies installed successfully!")

## Cell 3: Start Ollama Background Server Daemon (Official Kaggle Method)

In [ ]:
import os
import subprocess
import time
import httpx

# Start Ollama background process with OLLAMA_ORIGINS='*' (Allows external ngrok access)
env = os.environ.copy()
env['OLLAMA_ORIGINS'] = '*'
env['OLLAMA_HOST'] = '0.0.0.0:11434'

ollama_process = subprocess.Popen(['ollama', 'serve'], env=env)
print("⏳ Ollama server start ho raha hai, 5 seconds wait karein...")
time.sleep(5)

# Server connection test
try:
    res = httpx.get("http://127.0.0.1:11434/")
    print("✅", res.text)
    print("🚀 Ollama server LIVE ho chuka hai (OLLAMA_ORIGINS='*' enabled)!")
except Exception as e:
    print("⚠️ Re-checking server...", e)


## Cell 4: Pull Google Gemma 4 12B Model (100% Free & Tokenless)

In [ ]:
# Pull official Google Gemma 4 12B model
print("⏳ Pulling Gemma 4 12B Multimodal Model on Kaggle GPU...")
!ollama pull gemma4:12b

print("✅ Gemma 4 12B successfully cached on Kaggle GPU!")
!ollama list

## Cell 5: Compile Custom LabLens Clinical Persona

In [ ]:
modelfile_content = '''FROM gemma4:12b
PARAMETER temperature 0.3
PARAMETER top_p 0.9
PARAMETER num_ctx 8192

SYSTEM """
You are LabLens AI, an expert clinical review assistant and medical report interpreter powered by Google Gemma 4 12B.
Your core operational guidelines:
1. EvidenceTrace Grounding: Every observation must reference the specific test name, observed value, unit, and standard laboratory reference range.
2. Deterministic Accuracy: Explicitly flag whether measurements are Normal, Low, or High.
3. No Autonomous Diagnosing: Explain what tests measure and why they matter without declaring an independent diagnosis.
4. Questions for the Doctor: Conclude every report review with 3-4 specific, prioritized questions for the patient to ask their physician.
5. Always append: 'Disclaimer: Educational review assistant. Does not replace professional medical judgment.'
"""
'''

with open("Modelfile.clinical", "w", encoding="utf-8") as f:
    f.write(modelfile_content)

# Model compile karo
!ollama create lablens-clinical -f Modelfile.clinical
print("🎉 Gemma 4 12B LabLens Clinical Model successfully compiled!")
!ollama list

## Cell 6: Test Clinical Inference (EvidenceTrace & Review)

In [ ]:
import httpx

sample_prompt = (
    "PATIENT LAB REPORT:\n"
    "Hemoglobin: 10.4 g/dL (Reference: 13.0 - 17.0)\n"
    "Fasting Blood Sugar: 144 mg/dL (Reference: 70 - 99)\n"
    "Platelet Count: 135000 /mcL (Reference: 150000 - 450000)\n\n"
    "Please analyze this report with an EvidenceTrace table, explain findings, and provide doctor questions."
)

test_payload = {
    "model": "lablens-clinical",
    "prompt": sample_prompt,
    "stream": False
}

print("🩺 Querying lablens-clinical on Kaggle GPU...")
res = httpx.post("http://127.0.0.1:11434/api/generate", json=test_payload, timeout=90.0)
print("\n--- CLINICAL MODEL OUTPUT ---\n")
print(res.json().get("response", ""))

## Cell 7: Expose Kaggle GPU via Live Public Tunnel (ngrok)

In [ ]:
import os
from pyngrok import ngrok

# 1. Cleanly reset old tunnels
try:
    ngrok.kill()
except Exception:
    pass

# 2. Paste your free ngrok token below:
NGROK_TOKEN = "YOUR_NGROK_AUTHTOKEN_HERE"

# 3. Static domain (or leave None for dynamic):
NGROK_STATIC_DOMAIN = "proxy-service-editor.ngrok-free.dev"  # ya None

if NGROK_TOKEN != "YOUR_NGROK_AUTHTOKEN_HERE":
    os.system(f"ngrok config add-authtoken {NGROK_TOKEN}")
    try:
        ngrok.set_auth_token(NGROK_TOKEN)
    except Exception:
        pass

    # Connect tunnel to Ollama native port 11434
    try:
        if NGROK_STATIC_DOMAIN:
            tunnel = ngrok.connect(11434, domain=NGROK_STATIC_DOMAIN, host_header="localhost:11434")
        else:
            tunnel = ngrok.connect(11434, host_header="localhost:11434")
    except Exception as e:
        print(f"Static domain note: {e}. Opening standard tunnel...")
        tunnel = ngrok.connect(11434, host_header="localhost:11434")

    print(f"\n⚡ Public Cloud GPU URL: {tunnel.public_url}")
    print("👉 Set this in your local .env as CLOUD_GPU_URL!")
    print(f"   CLOUD_GPU_URL={tunnel.public_url}")
    print("\n✅ Your local LabLens AI backend will now route all Gemma 4 requests to this Kaggle GPU!")
else:
    print("ℹ️ Add your free NGROK_TOKEN above to expose your Kaggle GPU to the internet.")